In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ipywidgets import FloatProgress, Layout
from IPython.display import display
import micasense.imageset as imageset
import micasense.capture as capture
import os, glob
import multiprocessing

panelNames = None
useDLS = True

imagePath = '../../../data/udine/original_data/captures'
panelPath = '../../../data/udine/original_data/panel'
panelNames = glob.glob(os.path.join(panelPath,'IMG_0000_*.tif'))

outputPath = '../../../data/udine/preprocessing/stacks'
thumbnailPath = '../../../data/udine/preprocessing/rgb'

overwrite = False # Set to False to continue interrupted processing
generateThumbnails = True

# Allow this code to align both radiance and reflectance images; bu excluding
# a definition for panelNames above, radiance images will be used
# For panel images, efforts will be made to automatically extract the panel information
# but if the panel/firmware is before Altum 1.3.5, RedEdge 5.1.7 the panel reflectance
# will need to be set in the panel_reflectance_by_band variable.
# Note: radiance images will not be used to properly create NDVI/NDRE images below.
if panelNames is not None:
    panelCap = capture.Capture.from_filelist(panelNames)
    print('Panel captures loaded successfully.')
else:
    panelCap = None
    print('Panel captures not loaded.')

if panelCap is not None:
    if panelCap.panel_albedo() is not None:
        panel_reflectance_by_band = panelCap.panel_albedo()
        print('Panel reflectance loaded successfully.')
    else:
        panel_reflectance_by_band = [0.65]*len(panelCap.images) #inexact, but quick
        print('Panel reflectance not loaded. Using approximate reflectance.')
    panel_irradiance = panelCap.panel_irradiance(panel_reflectance_by_band)
    img_type = "reflectance"
else:
    if useDLS:
        img_type='reflectance'
        print('Using reflectance DLS.')
    else:
        img_type = "radiance"
        print('Using radiance.')


Panel captures loaded successfully.
Panel reflectance loaded successfully.


In [3]:
## This progress widget is used for display of the long-running process
f = FloatProgress(min=0, max=1, layout=Layout(width='100%'), description="Loading")
display(f)
def update_f(val):
    if (val - f.value) > 0.005 or val == 1: #reduces cpu usage from updating the progressbar by 10x
        f.value=val

%time imgset = imageset.ImageSet.from_directory(imagePath, progress_callback=update_f)
update_f(1.0)


FloatProgress(value=0.0, description='Loading', layout=Layout(width='100%'), max=1.0)

CPU times: user 19.5 ms, sys: 4.76 ms, total: 24.3 ms
Wall time: 173 ms


In [9]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import micasense.imageutils as imageutils
import micasense.plotutils as plotutils

alignment_img = glob.glob(os.path.join(imagePath,'IMG_0281_*.tif'))
cpt = capture.Capture.from_filelist(alignment_img)

## Alignment settings
match_index = 4 # Index of the band
                    # Blue: 0
                    # Green: 1
                    # Red: 2
                    # NIR: 3
                    # Red Edge: 4
                    # Blue-444: 5
                    # Green-531: 6
                    # Red-650: 7
                    # Red edge-705: 8
                    # Red edge-740: 9
max_alignment_iterations = 20
warp_mode = cv2.MOTION_HOMOGRAPHY # MOTION_HOMOGRAPHY or MOTION_AFFINE. For Altum images only use HOMOGRAPHY
pyramid_levels = 3 # for 10-band imagery we use a 3-level pyramid. In some cases

print("Alinging images. Depending on settings this can take from a few seconds to many minutes")
# Can potentially increase max_iterations for better results, but longer runtimes
warp_matrices, alignment_pairs = imageutils.align_capture(cpt,
                                                          ref_index = match_index,
                                                          max_iterations = max_alignment_iterations,
                                                          warp_mode = warp_mode,
                                                          pyramid_levels = pyramid_levels)

print("Finished Aligning, warp matrices={}".format(warp_matrices))


Alinging images. Depending on settings this can take from a few seconds to many minutes


IndexError: list index out of range

In [ ]:
import exiftool
import datetime

use_multi_process = True # set to False for single-process saving
overwrite_existing = False # skip existing files, set to True to overwrite

## This progress widget is used for display of the long-running process
f2 = FloatProgress(min=0, max=1, layout=Layout(width='100%'), description="Saving")
display(f2)
def update_f2(val):
    f2.value=val

if not os.path.exists(outputPath):
    os.makedirs(outputPath)
if generateThumbnails and not os.path.exists(thumbnailPath):
    os.makedirs(thumbnailPath)

# Save out geojson data so we can open the image capture locations in our GIS
#with open(os.path.join(outputPath,'imageSet.json'),'w') as f:
#    f.write(str(geojson_data))

# If we didn't provide a panel above, irradiance set to None will cause DLS data to be used
try:
    irradiance = panel_irradiance+[0]
except NameError:
    irradiance = None

start_time = datetime.datetime.now()

# Save all captures in the imageset as aligned stacks
imgset.save_stacks(warp_matrices,
                     outputPath,
                     thumbnailPath,
                     irradiance = irradiance,
                     multiprocess=use_multi_process, 
                     overwrite=overwrite_existing, 
                     progress_callback=update_f2)

end_time = datetime.datetime.now()
update_f2(1.0)

print("Saving time: {}".format(end_time-start_time))
print("Alignment+Saving rate: {:.2f} captures per second".format(float(len(imgset.captures))/float((end_time-start_time).total_seconds())))



In [ ]:
# Rename stacks and RGB files
for capture in imgset.captures:
    filename = capture.images[0].meta.get_item("File:FileName")[:8]

    for directory, suffix in zip((outputPath, thumbnailPath), ('.tif', '_rgb.jpg')):
        old_path = os.path.join(directory, f'{capture.uuid}{suffix}')
        new_path = os.path.join(directory, f'{filename}{suffix}')

        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            print(f'Renamed: {os.path.basename(old_path)} -> {os.path.basename(new_path)}')
